# Посимвольная генерация текста с помощью  RNN

In [10]:
import torch
from collections import Counter
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

Установка / импорт и загрузка файла:

In [2]:
# Colab cell 1: imports + download
# В Colab можно запускать как есть. Если не в Colab, строка wget тоже сработает в большинстве окружений.
!apt-get -y install -qq git >/dev/null 2>&1 || true

# скачиваем tiny shakespeare (raw)
!wget -q -O input.txt https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

print(text[:500])
print(len(text))

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor
1115394


Построение словаря символов (char ↔ idx) и кодирование всего текста в индексы:

In [5]:
chars = set()
for char in text:
    chars.add(char)

stoi = {char: i for i, char in enumerate(chars)}
itos = {i:char for i, char in enumerate(chars)}

def encode(string: str) -> list[int]:
    return [stoi[char] for char in string]

def decode(indices: list[int]) -> str:
    return "".join([itos[index] for index in indices])

data = torch.tensor(encode(text), dtype=torch.long)
print("Тензор данных shape:", data.shape)

Тензор данных shape: torch.Size([1115394])


Разбиение на последовательности, создание Dataset и DataLoader (BPTT-подход — последовательности фиксированной длины):

In [17]:
class CharDataset(Dataset):
    def __init__(self, data_tensor, seq_len):
        self.data = data_tensor
        self.seq_len = seq_len
        # число доступных стартов (только те, где хватит seq_len+1 символов)
        self.num_sequences = (len(self.data) - 1) // self.seq_len

    def __len__(self):
        return self.num_sequences

    def __getitem__(self, idx):
        start = idx * self.seq_len
        end = start + self.seq_len
        if end + 1 > len(self.data):
            # если не хватает символов — пропускаем (хотя drop_last=True должен помочь)
            end = len(self.data) - 1
        x = self.data[start:end]
        y = self.data[start + 1:end + 1]

        # чтобы точно избежать ошибок — дропаем слишком короткие куски
        if len(x) < self.seq_len or len(y) < self.seq_len:
            x = torch.zeros(self.seq_len, dtype=torch.long)
            y = torch.zeros(self.seq_len, dtype=torch.long)
        return x, y

In [19]:
SEQ_LEN = 100
BATCH_SIZE = 64

dataset = CharDataset(data, seq_len=SEQ_LEN)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

In [20]:
class SimpleRNN(nn.Module):
    def __init__(self, vocab_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size

        # Вход (one-hot) сначала проецируем в скрытое состояние
        self.W_xh = nn.Parameter(torch.randn(vocab_size, hidden_size) * 0.01)
        # Предыдущее скрытое состояние → новое скрытое состояние
        self.W_hh = nn.Parameter(torch.randn(hidden_size, hidden_size) * 0.01)
        # Смещение для скрытого состояния
        self.b_h = nn.Parameter(torch.zeros(hidden_size))

        # Преобразование скрытого состояния в логиты символов
        self.W_hy = nn.Parameter(torch.randn(hidden_size, vocab_size) * 0.01)
        self.b_y = nn.Parameter(torch.zeros(vocab_size))

    def forward(self, x, h_prev):
        """
        x: (batch, seq_len) — индексы символов
        h_prev: (batch, hidden_size)
        """
        batch_size, seq_len = x.shape
        outputs = []
        h = h_prev

        for t in range(seq_len):
            x_t = F.one_hot(x[:, t], num_classes=self.W_xh.shape[0]).float()  # (batch, vocab_size)
            h = torch.tanh(x_t @ self.W_xh + h @ self.W_hh + self.b_h)
            y = h @ self.W_hy + self.b_y
            outputs.append(y.unsqueeze(1))  # добавляем размерность seq_len

        outputs = torch.cat(outputs, dim=1)  # (batch, seq_len, vocab_size)
        return outputs, h


In [21]:
vocab_size = len(stoi)
hidden_size = 256  # например, можно увеличить для лучшего качества
print("Размер словаря:", vocab_size)


Размер словаря: 65


In [22]:
num_epochs = 10
lr = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'

model = SimpleRNN(vocab_size, hidden_size).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=lr)

def generate_text(model, start_text="ROMEO:", length=300, temperature=1.0):
    """Генерация текста символ за символом"""
    model.eval()
    input_seq = torch.tensor(encode(start_text), dtype=torch.long).unsqueeze(0).to(device)
    h = torch.zeros(1, model.hidden_size).to(device)
    generated = list(start_text)

    for _ in range(length):
        with torch.no_grad():
            logits, h = model(input_seq, h)
            logits = logits[:, -1, :] / temperature
            probs = torch.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1).item()

        generated.append(itos[next_id])
        input_seq = torch.tensor([[next_id]], dtype=torch.long).to(device)

    return ''.join(generated)

# основной цикл обучения
for epoch in range(num_epochs):
    model.train()
    total_loss = 0

    for xb, yb in dataloader:
        xb, yb = xb.to(device), yb.to(device)
        h0 = torch.zeros(xb.size(0), hidden_size).to(device)
        logits, _ = model(xb, h0)

        # приводим logits и цели к нужной форме для CrossEntropy
        loss = criterion(logits.view(-1, vocab_size), yb.view(-1))

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # обрезаем градиенты
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(dataloader)
    print(f"Эпоха {epoch+1}/{num_epochs}, Loss: {avg_loss:.4f}")

    # каждые несколько эпох — генерация текста
    if (epoch + 1) % 2 == 0:
        sample = generate_text(model, start_text="ROMEO:", length=400, temperature=0.8)
        print("\n=== Сгенерированный текст ===\n")
        print(sample)
        print("\n=============================\n")


Эпоха 1/10, Loss: 3.3429
Эпоха 2/10, Loss: 2.9094

=== Сгенерированный текст ===

ROMEO::OW aa he thee ot tod taev tireunthom d ttire yrr gee winass il oosh nouir. soaun .han, wed whe.iwerer nree on iit ety saln mer sause  ois his thans aan ss
aostin I toth

r she toc en os ure d whas de,oOl

UI boton ton th not balhe kamee heds fas eh re nnt  or ne itkotxode ofe wimn toy ?as soi t eweot
 teelator lran sash rirok ron,Tle gereobsulne  teo oo toep yimldrcatin:
EL inde danitaet ne be mh


Эпоха 3/10, Loss: 2.5439
Эпоха 4/10, Loss: 2.3998

=== Сгенерированный текст ===

ROMEO:
Nevendey hacd bied thas uase are ne het and mer thit hatlasers tor tanges he fouk ae the ther mait mt an.

You may lodt aot ano bode soun tea. 
oos antouf the Poad thed sare.

Sind wad, are in the he lallant of aadinn sis nome by thy und andt or maeghin sme bewesend corl eat you touin mide the herandy sirre, sot Oe peearidesind anty weest Cave ccild tou tould cas pheshat larr at werpe anithur wan


Эпоха 5/10, Loss: 